# 📡 TelecomX — Análisis de Evasión de Clientes (Churn)
### Challenge 2 · Oracle Next Education × Alura Latam

**Objetivo:** Identificar los factores que llevan a la evasión de clientes en Telecom X, aplicando ETL y Análisis Exploratorio de Datos (EDA) para generar insights estratégicos.

---

## 📥 Paso 1 — Extracción de datos desde la API

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

url_api = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json'
df_raw = pd.read_json(url_api)

print('Datos extraidos correctamente')
print(f'Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}')
df_raw.head()

---
## 🔍 Paso 2 — Conocer el conjunto de datos

In [ ]:
print('=== Información general del dataset ===')
df_raw.info()
print('\n=== Columnas disponibles ===')
print(list(df_raw.columns))

In [ ]:
# Normalizar columnas anidadas si existen
cols_anidadas = [col for col in df_raw.columns if df_raw[col].apply(lambda x: isinstance(x, dict)).any()]
if cols_anidadas:
    print(f'Columnas anidadas detectadas: {cols_anidadas}')
    df = df_raw.copy()
    for col in cols_anidadas:
        expandida = pd.json_normalize(df[col])
        expandida.columns = [f'{col}_{c}' for c in expandida.columns]
        df = pd.concat([df.drop(columns=[col]), expandida], axis=1)
    print(f'Dataset normalizado: {df.shape[0]} filas x {df.shape[1]} columnas')
else:
    df = df_raw.copy()
    print('No hay columnas anidadas.')
df.head()

---
## 🔎 Paso 3 — Comprobación de incoherencias en los datos

In [ ]:
print('=== Valores nulos por columna ===')
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else 'No hay valores nulos')
print(f'\n=== Filas duplicadas: {df.duplicated().sum()} ===')
print('\n=== Tipos de datos ===')
print(df.dtypes)

In [ ]:
cols_object = df.select_dtypes(include='object').columns
print('=== Valores únicos en columnas categóricas ===')
for col in cols_object:
    print(f'  {col}: {df[col].unique()[:10]}')

---
## 🛠️ Paso 4 — Manejo de inconsistencias (Limpieza)

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()
cargos_cols = [c for c in df_clean.columns if 'cargo' in c.lower() or 'charge' in c.lower() or 'total' in c.lower()]
for col in cargos_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
for col in df_clean.select_dtypes(include='number').columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col].replace('nan', np.nan, inplace=True)
    df_clean[col].fillna('Desconocido', inplace=True)
print(f'Dataset limpio: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas')
print(f'Nulos restantes: {df_clean.isnull().sum().sum()}')

In [ ]:
mapeo_binario = {'Yes': 1, 'No': 0, 'Si': 1, 'yes': 1, 'no': 0}
churn_col = [c for c in df_clean.columns if 'churn' in c.lower() or 'evas' in c.lower()]
print(f'Columna churn detectada: {churn_col}')
if churn_col:
    col = churn_col[0]
    # Mapear Yes/No a 1/0
    df_clean[col] = df_clean[col].map(mapeo_binario)
    # Forzar conversión numérica por si quedaron valores no mapeados
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(int)
    print(f'Columna {col} convertida. Valores únicos: {df_clean[col].unique()}')
df_clean.head()

---
## ➕ Paso 5 — Columna Cargos Diarios (opcional)

In [ ]:
mensual_col = [c for c in df_clean.columns if 'mensual' in c.lower() or 'monthly' in c.lower()]
if mensual_col:
    col = mensual_col[0]
    df_clean['Cargos_Diarios'] = (df_clean[col] / 30).round(4)
    print(f'Cargos_Diarios creada desde: {col}')
    print(df_clean['Cargos_Diarios'].describe())
else:
    print('Columnas disponibles:', list(df_clean.columns))

---
## 📊 Paso 6 — Análisis Descriptivo

In [ ]:
display(df_clean.describe())

In [ ]:
if churn_col:
    col = churn_col[0]
    conteo = df_clean[col].value_counts()
    pct = df_clean[col].value_counts(normalize=True) * 100
    print('=== Distribución de evasión ===')
    for val in conteo.index:
        etiqueta = 'Evadieron' if val == 1 else 'Se quedaron'
        print(f'  {etiqueta}: {conteo[val]} clientes ({pct[val]:.1f}%)')

---
## 📈 Paso 7 — Visualizaciones

### Gráfico 1 — Distribución de Evasión (Pie)

In [ ]:
if churn_col:
    col = churn_col[0]
    conteo = df_clean[col].value_counts()
    etiquetas = ['Se quedaron' if i == 0 else 'Evadieron' for i in conteo.index]
    explode = [0.05 if i == conteo.idxmax() else 0 for i in conteo.index]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(conteo.values, labels=etiquetas, autopct='%1.1f%%',
           colors=['#4CAF50', '#F44336'][:len(conteo)],
           startangle=90, explode=explode, shadow=True,
           textprops={'fontsize': 12})
    ax.set_title('Distribución de Evasión de Clientes (Churn)', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('grafico_churn_distribucion.png', dpi=150, bbox_inches='tight')
    plt.show()

### Gráfico 2 — Evasión por Variables Categóricas

In [ ]:
if churn_col:
    churn = churn_col[0]
    cat_cols = [c for c in df_clean.select_dtypes(include='object').columns
                if c.lower() not in ['customerid', 'customer_id', 'id']][:6]
    if cat_cols:
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        axes = axes.flatten()
        for i, col in enumerate(cat_cols):
            tabla = df_clean.groupby([col, churn]).size().unstack(fill_value=0)
            tabla.plot(kind='bar', ax=axes[i], color=['#4CAF50', '#F44336'], edgecolor='white', width=0.7)
            axes[i].set_title(f'Evasión por {col}', fontsize=11, fontweight='bold')
            axes[i].set_xlabel('')
            axes[i].set_ylabel('Clientes')
            axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')
            axes[i].legend(['No evadieron', 'Evadieron'], fontsize=8)
            axes[i].spines['top'].set_visible(False)
            axes[i].spines['right'].set_visible(False)
        for j in range(len(cat_cols), len(axes)):
            axes[j].set_visible(False)
        plt.suptitle('Evasión por Variables Categóricas', fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.savefig('grafico_churn_categoricas.png', dpi=150, bbox_inches='tight')
        plt.show()

### Gráfico 3 — Evasión por Variables Numéricas (Histogramas)

In [ ]:
if churn_col:
    churn = churn_col[0]
    num_cols = [c for c in df_clean.select_dtypes(include='number').columns
                if c != churn and 'id' not in c.lower()][:4]
    if num_cols:
        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        axes = axes.flatten()
        for i, col in enumerate(num_cols):
            for val, color, label in [(0, '#4CAF50', 'No evadieron'), (1, '#F44336', 'Evadieron')]:
                subset = df_clean[df_clean[churn] == val][col].dropna()
                axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
            axes[i].set_title(f'Distribución de {col} por Churn', fontsize=11, fontweight='bold')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Frecuencia')
            axes[i].legend()
            axes[i].spines['top'].set_visible(False)
            axes[i].spines['right'].set_visible(False)
        for j in range(len(num_cols), len(axes)):
            axes[j].set_visible(False)
        plt.suptitle('Variables Numéricas por Evasión', fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.savefig('grafico_churn_numericas.png', dpi=150, bbox_inches='tight')
        plt.show()

### Gráfico 4 — Cargos Mensuales vs Tiempo de Contrato (Dispersión)

In [ ]:
if churn_col:
    churn = churn_col[0]
    tenure_col = [c for c in df_clean.columns if 'tenure' in c.lower() or 'tiempo' in c.lower() or 'meses' in c.lower()]
    mensual_col2 = [c for c in df_clean.columns if 'mensual' in c.lower() or 'monthly' in c.lower()]
    if tenure_col and mensual_col2:
        t_col = tenure_col[0]
        m_col = mensual_col2[0]
        fig, ax = plt.subplots(figsize=(9, 6))
        for val, color, label in [(0, '#4CAF50', 'No evadieron'), (1, '#F44336', 'Evadieron')]:
            subset = df_clean[df_clean[churn] == val]
            ax.scatter(subset[t_col], subset[m_col], alpha=0.4, color=color, label=label, s=15)
        ax.set_title('Cargos Mensuales vs Tiempo de Contrato por Churn', fontsize=13, fontweight='bold')
        ax.set_xlabel('Tiempo de contrato (meses)')
        ax.set_ylabel('Cargos mensuales ($)')
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig('grafico_scatter_churn.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('Columnas disponibles:', list(df_clean.columns))

### Gráfico 5 — Matriz de Correlación (Extra)

In [ ]:
num_df = df_clean.select_dtypes(include='number')
if num_df.shape[1] > 1:
    corr = num_df.corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(corr.columns, fontsize=9)
    for i in range(len(corr)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
    ax.set_title('Matriz de Correlación', fontsize=13, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig('grafico_correlacion.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 📋 Paso 8 — Informe Final

### 🔹 Introducción

Telecom X enfrenta una alta tasa de cancelación de clientes (churn). El presente análisis aplica ETL y EDA para identificar los factores que influyen en la evasión y generar insights estratégicos.

### 🔹 Limpieza y Tratamiento de Datos

- **Extracción:** Datos cargados desde la API oficial en formato JSON.
- **Transformación:** Normalización de columnas anidadas, eliminación de duplicados, tratamiento de nulos, estandarización de strings y conversión de variables binarias.
- **Carga:** Dataset limpio listo para análisis con columna adicional `Cargos_Diarios`.

### 🔹 Análisis Exploratorio de Datos

In [ ]:
if churn_col:
    churn = churn_col[0]
    total = len(df_clean)
    evadieron = df_clean[churn].sum()
    tasa = evadieron / total * 100
    print('=== RESUMEN EJECUTIVO ===')
    print(f'  Total clientes analizados : {total:,}')
    print(f'  Clientes que evadieron    : {int(evadieron):,} ({tasa:.1f}%)')
    print(f'  Clientes que se quedaron  : {int(total - evadieron):,} ({100 - tasa:.1f}%)')
    mensual_col3 = [c for c in df_clean.columns if 'mensual' in c.lower() or 'monthly' in c.lower()]
    if mensual_col3:
        m = mensual_col3[0]
        print(f'\n  Cargo mensual prom. (evadieron)   : ${df_clean[df_clean[churn]==1][m].mean():.2f}')
        print(f'  Cargo mensual prom. (se quedaron) : ${df_clean[df_clean[churn]==0][m].mean():.2f}')

### 🔹 Conclusiones e Insights

1. **Tasa de churn:** El **25.7%** de los clientes cancelaron el servicio (1,869 de 7,267), lo que representa una pérdida significativa de ingresos para Telecom X.

2. **Cargos mensuales:** Los clientes que evadieron pagaban en promedio **$74.44/mes**, mientras que los que se quedaron pagaban **$61.35/mes**. A mayor cargo mensual, mayor riesgo de churn.

3. **Contratos:** Los clientes con contrato mes a mes presentan mayor tasa de evasión que los que tienen contratos anuales o bianuales.

4. **Tenure:** Los clientes con menor antigüedad tienen mayor probabilidad de churn — la fidelización temprana es clave para retener clientes.

5. **Correlación:** Existe una correlación negativa entre el tiempo de contrato y la evasión: a mayor antigüedad, menor probabilidad de cancelar.

### 🔹 Recomendaciones

1. **Fidelización temprana:** Implementar programas de retención dirigidos a clientes con menos de 12 meses de contrato, que son el grupo de mayor riesgo.
2. **Incentivos para contratos largos:** Ofrecer descuentos o beneficios exclusivos para clientes que migren de contrato mensual a anual.
3. **Revisión de precios:** Los clientes con cargos mensuales altos (+$74) evaden más. Evaluar planes competitivos o descuentos por fidelidad.
4. **Atención proactiva:** Contactar clientes en riesgo (bajo tenure + cargo alto) antes de que tomen la decisión de cancelar.
5. **Modelos predictivos:** Con los datos limpios generados en este análisis, el equipo de Data Science puede construir modelos de clasificación (Random Forest, Regresión Logística) para predecir churn en tiempo real.